<a href="https://colab.research.google.com/github/poorani3010-code/Multilingual-Medical-Chatbot./blob/main/Medical_Chatbot_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#install the necessary libraries for our chatbot
!pip install transformers datasets sentencepiece sacremoses streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 56.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
data_url="https://raw.githubusercontent.com/poorani3010-code/Multilingual-Medical-Chatbot./refs/heads/main/data/medquad_sample.csv"
df=pd.read_csv(data_url)
print("Data loaded successfully!")
df.head()

Data loaded successfully!


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [3]:
df['question']=df['question'].str.lower().str.strip()
print("Data cleaning complete.")
df.head()

Data cleaning complete.


,question,answer,source,focus_area
0,what is (are) glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,what causes glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,what are the symptoms of glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,what are the treatments for glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,what is (are) glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [4]:
df['text']="Question:"+df['question']+"Answer:"+df['answer']
print(df['text'].iloc[0])

Question:what is (are) glaucoma ?Answer:Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the optic nerve and other parts of the eye and result in loss of vision. Open-angle Glau

In [5]:
from transformers import AutoTokenizer
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!


In [6]:
from datasets import Dataset
dataset = Dataset.from_pandas(df[['text']])
def tokenize_function(examples):
  return  tokenizer(examples['text'], truncation=True, padding ="max_length", max_length=128)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

In [10]:
from transformers import TrainingArguments,Trainer, DataCollatorForLanguageModeling, AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
data_collator = DataCollatorForLanguageModeling (tokenizer=tokenizer,mlm=False)
training_args=TrainingArguments(output_dir="./medical_model_results",num_train_epochs=3,per_device_train_batch_size=4,save_steps=100,logging_steps=10)
trainer=Trainer(model=model,args=training_args,train_dataset=tokenized_datasets,data_collator=data_collator,)
print ("Ready to Train")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready to Train


In [11]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Step,Training Loss
10,3.100190
20,2.676315
30,2.491620


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=39, training_loss=2.6718758803147535, metrics={'train_runtime': 262.2398, 'train_samples_per_second': 0.561, 'train_steps_per_second': 0.149, 'total_flos': 4801327792128.0, 'train_loss': 2.6718758803147535, 'epoch': 3.0})

In [13]:
model.save_pretrained("./final_medical_model")
tokenizer.save_pretrained("./final_medical_model")
print("Model saved successfully!It is now a permanent file in your collab.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!It is now a permanent file in your collab.


In [15]:
model.save_pretrained("./medical_chatbot_model")
tokenizer.save_pretrained("./medical_chatbot_model")
print("Sucess! Your medical AI is now saved in the 'medical_chtbot_model'folder.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sucess! Your medical AI is now saved in the 'medical_chtbot_model'folder.


In [17]:
from transformers import pipeline
medical_chat=pipeline("text-generation",model="./medical_chatbot_model",tokenizer="./medical_chatbot_model")
prompt="Question:what is (are)glaucoma?Answer"
result=medical_chat(prompt,max_length=100,num_return_sequences=1)
print(result[0]['generated_text'])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:what is (are)glaucoma?Answer:Yes.Glaucoma is a type of inflammation that causes inflammation and damages the bone, which causes inflammation in the joints, and joints. Glaucoma typically affects the blood vessels in the bone, but isn't found in the kidneys. If you have untreated glaucoma, you may need to treat it. You can prevent the symptoms of the disease by using the anti-inflammatory anti-inflammatory anti-inflammatory anti-inflammatory anti-inflammatory anti-inflammatory anti-inflammatory anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflammation anti-inflam

In [20]:
import streamlit as st
from transformers import pipeline
st.title("Multilingual Medical Chatbot")
st.write("Ask a medical question below:")
def load_chatbot():
  return pipeline("text-generation",model="./medical_chtbot_model")
  chat_pipeline=load_chatbot()
  user_input=st.text_input("Your Question:")
  if user_input:
    with st.spinner("Thinking..."):
      promt=f"Question:{user_input} Answer:"
      response=chat_pipeline(prompt,max_length=100)[0]['generated_text']
      st.success(response)

2026-03-07 17:31:59.757 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 17:31:59.940 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-07 17:31:59.941 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 17:31:59.942 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 17:31:59.943 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 17:31:59.943 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-07 17:31:59.944 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [23]:
import shutil
from google.colab import files
shutil.make_archive("medical_chatbot_model", "zip", "./medical_chatbot_model")
files.download("medical_chatbot_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>